# Substitution Ciphers

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import string

In [ ]:
data_dir = Path().resolve().parents[0] / "data"

## Introduction


Substitution ciphers are classical encryption techniques in which each element of a plaintext message, typically a letter, is replaced by another symbol according to a fixed mapping defined by a **key**. 

The replacement remains consistent throughout the message.​

The receiver decrypts the message by applying the **inverse substitution**, recovering the original plaintext.

Although historically important, substitution ciphers are vulnerable to cryptanalytic methods such as **frequency analysis**, which exploit statistical patterns in natural language and also as **brute force attacks**, due to the limited key space.

## Caesar Cipher

The method is named after Julius Caesar, who used it in his private correspondence. Each letter in the plaintext is replaced by a letter shifted by some fixed number of positions down the alphabet. 

Its a extremely simple encryption rule, it has only 26 possibile keys!

![Caesar Encryption](images/caesar.png)

**Types of attack of Caesar Cipher:**

**Brute force**: ​The English alphabet is 26 letters long, meaning that only 26 shifts are possible. Hence, you can try all possibilities and check whether the resulting plaintext makes sense.​

### Encrypt

In [ ]:
def caesar_encrypt(plaintext, shift=0):
    ''' Encrypt `plaintext` (str) as a caesar cipher with a given `shift` (int)
        and return the `ciphertext` (str) '''
    # define plaintext alphabet
    plain_alphabet = string.ascii_lowercase
    # define ciphertext alphabet
    cipher_alphabet = plain_alphabet[shift:] + plain_alphabet[:shift]
    # build a mapping that trasforms a plaintext letter into a ciphertext letter
    mapping = dict(zip(plain_alphabet, cipher_alphabet))
    # apply the transformation to the plaintext
    ciphertext = ''.join(mapping.get(char, char) for char in plaintext)

    return ciphertext

In [ ]:
# code snippet to test the implementation of the decoder
plaintext = 'hello!'
ciphertext = caesar_encrypt(plaintext, shift=4)

print(plaintext, '->', ciphertext) # expected output 'hello! -> lipps!'

### Decrypt

Decryption works by shifting each letter of the ciphertext **backward** by the same number of positions $k$ used during encryption. If the cipher used a shift of 𝑘, decryption applies a shift of $−k$ to recover the original plaintext.

In [ ]:
def caesar_decrypt(ciphertext, shift=0):
    ''' Decrypt `ciphertext` (str) as a caesar cipher with a given `shift` (int)
        and return the `plaintext` (str) '''
    # define plaintext alphabet
    plain_alphabet = string.ascii_lowercase
    # define ciphertext alphabet
    cipher_alphabet = plain_alphabet[shift:] + plain_alphabet[:shift]
    # build a mapping that trasform a plaintext letter into a ciphertext letter
    mapping = dict(zip(cipher_alphabet, plain_alphabet))
    # apply the transformation to the plaintext
    plaintext = ''.join(mapping.get(char, char) for char in ciphertext)
    return plaintext

In [ ]:
# code snippet to test the implementation of the decoder
ciphertext = 'lipps!' # 'hello!' encoded with shift=4
plaintext = caesar_decrypt(ciphertext, shift=4)

print(ciphertext, '->', plaintext)

### Ciphertext

In [ ]:
# Load ciphertext
with open(data_dir / 'ciphertext_caesar.txt', mode='r', encoding='utf-8') as f:
    ciphertext = f.read()

# print first 500 characters
print(ciphertext[:500])

### Brute Force

A **brute force attack** consists of trying **all possible keys** (i.e., shifts) of the Caesar cipher.  
Since the alphabet has **26 letters**, there are **25 possible keys** (let us remove the null shift which keep the plaintext plain).

The attacker decrypts the ciphertext using each possible shift and checks which result produces a **meaningful plaintext message**.

In [ ]:
# Perform a Brute Force attack
print('shift  ciphertext')
for i in range(26):
    print(f'{i:4}   {caesar_decrypt(ciphertext, i)[:70]}')

In [ ]:
# select the right shift
shift = 18

# decrypt ciphertext
plaintext = caesar_decrypt(ciphertext, shift)
print(plaintext)

## Simple Substitution Cipher

A Simple Substitution Cipher replaces each plaintext character with a **different ciphertext character**. As in the Caesar cipher, the plaintext and ciphertext use the **same alphabet**.

The mapping between plaintext letters and ciphertext letters can be **any permutation of the alphabet**, giving $26! \approx 10^{26} \approx 2^{88}$ possible substitution keys.

The receiver **decrypts** the text by performing the inverse substitution.

![Simple Substitution](images/simple_substitution.png)

**Type of attacks of a Substitution Cipher:**

1) **Brute Force**: consists of trying every possible key (i.e., shift). Even if we assume that each attempt takes only 1 nanosecond, checking all keys would require **more than $10^9$ years**. Since the number of possible substitution keys is $26! \approx 10^26$, it is not feasible for modern computers to test all possibilities.

2) **Frequency Analysis**: A substitution cipher keeps the **statistical characteristics of the language**, such as how often each letter appears. This property allows an attacker to estimate the plaintext by studying the frequency of letters in the ciphertext.

For sufficiently long ciphertexts (so that the statistics are meaningful), a possible approach is:
- Replace the **most frequent letter in the ciphertext** with the **most frequent letter of the language**
- Replace the **second most frequent ciphertext letter** with the **second most frequent letter**
- Continue this process for the remaining letters

However, the observed frequencies in the ciphertext will not perfectly match the expected probabilities of the language. As a result, the automatic substitutions will only partially recover the correct mapping of the substitution cipher, and additional manual adjustments are usually required to obtain the correct plaintext.

### Encryption

In [ ]:
def simple_encrypt(plaintext, mapping):
    ''' Encrypt `plaintext` (str) as a simple substitution cipher with a given
       `mapping` (dict) from plaintext letters to ciphertext letters '''
    ciphertext = ''.join(mapping.get(char, char) for char in plaintext)
    return ciphertext

In [ ]:
# code snippet to test the implementation of the encryption function
plaintext = 'hello!'
mapping = {'h': 'a', 'e': 'p', 'l': 'w', 'o': 'q'}

ciphertext = simple_encrypt(plaintext, mapping)

print(plaintext, '->', ciphertext) # expected output 'hello! -> apwwq!'

### Decryption

In [ ]:
def simple_decrypt(ciphertext, mapping):
    ''' Decrypt `ciphertext` (str) as a simple substitution cipher with a given
       `mapping` (dict) from plaintext letters to ciphertext letters '''
    inv_mapping = dict(zip(mapping.values(), mapping.keys()))
    plaintext = ''.join(inv_mapping.get(char, char) for char in ciphertext)
    return plaintext

In [ ]:
mapping = {'h': 'a', 'e': 'p', 'l': 'w', 'o': 'q'}  # previous mapping
ciphertext = 'apwwq!'

plaintext = simple_decrypt(ciphertext, mapping)

print(ciphertext, '->', plaintext)  # expected output 'apwwq! -> hello!'

### Frequency Analysis Attack

#### English Letters Distribution

In [ ]:
def letter_distribution(text: str) -> dict[str, float]:
    ''' Return the `distribution` (dict) of the letters in `text` (str) '''

    alphabet = string.ascii_lowercase
    hist = np.empty(len(alphabet), dtype=int)  # stores letter count
    for i, letter in enumerate(alphabet):      # iterate over every letter
        hist[i] = text.lower().count(letter)   # count letter occurence
    hist = map(float, hist / np.sum(hist))     # convert counts into prob.
    distribution = dict(zip(alphabet, hist))
    return distribution

In [ ]:
# code snippet to test the implementation of `letter_distribution`
text = 'hello world!'

letter_distribution(text)
# expected ouput:
# {'d': 0.1, 'e': 0.1, 'h': 0.1, 'l': 0.3, 'o': 0.2, 'r': 0.1, 'w': 0.1, ...}

In [ ]:
def plot_distribution(distribution, ax=None, title=None):
    ''' plot a letter distribution'''
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(np.arange(len(distribution)), list(distribution.values()), 1)
    ax.set(xticks=np.arange(len(distribution)), xticklabels=distribution.keys())
    ax.set(xlabel='letter', ylabel='probability', title=title)
    ax.grid()

In [ ]:
# load normal text to get distributions
with open(data_dir / 'wikipedia_cybersecurity.txt', mode='r') as f:
    text = f.read()

print(text[:400])

In [ ]:
eng_dist = letter_distribution(text)
eng_dist

In [ ]:
plot_distribution(eng_dist, title='English letter distribution')

In [ ]:
# store the distribution as a pickle file
with open(data_dir / 'eng_letter_distribution.pkl', mode='wb') as f:
    pickle.dump(eng_dist, f)

#### Attack

In [ ]:
# Load ciphertext
with open(data_dir / 'ciphertext_simple.txt', mode='r', encoding='utf-8') as f:
    ciphertext = f.read()

print(ciphertext[:400])

In [ ]:
ciphertext_dist = letter_distribution(ciphertext)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(6, 6))
plot_distribution(eng_dist, ax=axs[0], title='English letters distribution')
plot_distribution(ciphertext_dist, ax=axs[1], title='cipertext distribution')
fig.tight_layout()

In [ ]:
# function to sort alphabet letters by frequency
def sort_keys_by_value(d):
    """ returns dictionary keys (list) sorted by value"""
    return sorted(d, key=lambda k: d[k])

# sort alphabet letters by frequency in the English language (most common first)
plain_letters_sorted = sort_keys_by_value(eng_dist)

# sort alphabet letters by frequency in the English language (most common first)
cipher_letters_sorted = sort_keys_by_value(ciphertext_dist)

# create a mapping from plain alphabet to cipher alphabet
# this is a first tentative substitution mapping based on frequency analysis
mapping = dict(zip(plain_letters_sorted, cipher_letters_sorted))

print(' plain:', *mapping.keys())
print('cipher:', *mapping.values())

In [ ]:
plaintext = simple_decrypt(ciphertext, mapping)
print('ciphertext:', ciphertext[:100])
print(' plaintext:', plaintext[:100])

In [ ]:
#  ciphertext: ywbvfgmng ... pvycmsgmng
#  1st guess: ulaintext ... diuceotext
#             |             | || |
#  2nd guess: Plaintext ... CiPHeRtext

mapping['u'], mapping['p'] = mapping['p'], 'y'

mapping['c'], mapping['h'] = mapping['h'], 'c'
mapping['d'], mapping['c'] = mapping['c'], 'p'
mapping['o'], mapping['r'] = mapping['r'], 's'

print(' plain:', *mapping.keys())
print('cipher:', *mapping.values())

In [ ]:
plaintext = simple_decrypt(ciphertext, mapping)
print('ciphertext:', ciphertext[:100])
print(' plaintext:', plaintext[:100])

In [ ]:
# ciphertext: vf pseygdtsbyce, b lxolgvgxgvdf pvycms vl b kmgcdj dr mfpseygvft
#  2nd guess: in crgptobraphg, a smfstitmtion cipher is a uethod oy encrgptinb
#                  |   |    |     ||    |                 |       |     |    |
#  3rd guess: in crYptoGraphY, a sUBstitUtion cipher is a Method oF encrYptinG

mapping['g'], mapping['y'] = mapping['y'], 'e' # y as last
mapping['b'], mapping['g'] = mapping['g'], 't'
mapping['m'], mapping['u'] = mapping['u'], mapping['m'] # swap 'm' and 'u'

print(' plain:', *mapping.keys())
print('cipher:', *mapping.values())

In [ ]:
plaintext = simple_decrypt(ciphertext, mapping)
print('ciphertext:', ciphertext[:100])
print(' plaintext:', plaintext[:100])

In [ ]:
# ciphertext: dr mfpseygvft vf ucvpc
#  3rd guess: ob encrypting in vhich
#              |               |
#  4th guess: oF encrypting in Which
mapping['b'], mapping['f'] = mapping['f'], 'r'
mapping['v'], mapping['w'] = mapping['w'], 'u'

print(' plain:', *mapping.keys())
print('cipher:', *mapping.values())

In [ ]:
plaintext = simple_decrypt(ciphertext, mapping)
print('ciphertext:\n', ciphertext[:3000], '\n')
print('plaintext:\n', plaintext[:3000])

In [ ]:
# ciphertext: hxvgm ... "imosbl"
#  4th guess: zuite ... "jebras"
#             |          |
#  5th guess: Quite ... "Zebras"
mapping['z'], mapping['q'] = mapping['q'], 'h'
mapping['j'], mapping['z'] = mapping['z'], 'i'

print('   plain:', *mapping.keys())
print('ciphered:', *mapping.values())

In [ ]:
mapping_sorted = dict(sorted(mapping.items())) # sort mapping by letter
print('   plain:', *mapping_sorted.keys())
print('ciphered:', *mapping_sorted.values())

In [ ]:
plaintext = simple_decrypt(ciphertext, mapping)
print(plaintext[:4000])

## Affine cipher

The Affine cipher is a classical substitution cipher that encrypts letters using a simple mathematical formula -- an affine transformation.
The Affine cipher itself does not have a standalone history but it is more of a **mathematical generalization** of the Caesar Cipher. It is mostly employed in education to teach **modular arithmetic** in a concrete and engaging example.

### Encryption

To apply a mathematical formula to letter, we must first convert letters to numbers. This is simply done by converting each letter with tits index in the alphabet.

| A | B | C | D | ... | X  |  Y |  Z |
|---|---|---|---|-----|----|----|----|
| 0 | 1 | 2 | 3 | ... | 23 | 24 | 25 |

Then the each letter is encrypted using:
$$ y = {\rm Enc}_k(x) = a \cdot x + b \, ({\rm mod}\; m)$$
where:
- $x$ is the numerical value of the plaintext letter
- $a$ and $b$ are two integers representing the key
- $m$ is the lenght of the alphabet
- $y$ is the numerical value of the ciphertext letter

**Example**
Let us use the key $(a, b) = (3, 1)$, and the plaintext "*hello*".

| $x$ letter | $x$ number |            encryption              | $y$ number | $y$ letter |
|:----------:|:----------:|:----------------------------------:|:----------:|:----------:|
| h          |          7 | $3 \cdot  7 + 1 \,({\rm mod}\;26)$ |         22 |          w | 
| e          |          4 | $3 \cdot  4 + 1 \,({\rm mod}\;26)$ |         13 |          n | 
| l          |         11 | $3 \cdot 11 + 1 \,({\rm mod}\;26)$ |          8 |          i | 
| l          |         11 | $3 \cdot 11 + 1 \,({\rm mod}\;26)$ |          8 |          i | 
| o          |         14 | $3 \cdot 14 + 1 \,({\rm mod}\;26)$ |         17 |          r | 


Note that:
- the **Caesar cipher** is an Affine cipher with $a$ = 1.
- $a$ and $b$ are two integers between 0 and 25. Since the formula is modulo $m$, any larger value will behave exactly as a value smaller than $m$. For example, using $a=28$ and $b=41$ is equivalent to using $a=4$ and $b=15$.

In [ ]:
def affine_encrypt(plaintext, a, b):
    ''' Encrypt `plaintext` (str) as a Vigenere cipher with a given `key` made
       of two values `a` (int) and `b` (int) and return the corresponding
       ciphertext (str) '''
    # CODE HERE
    return ciphertext

In [ ]:
plaintext = 'hello world!'
a, b = 3, 1

ciphertext = affine_encrypt(plaintext, a, b)
print(plaintext, '->', ciphertext)
# expected output 'hello world! -> wniir praik!'

### Decryption

The **decryption** is the inverse function of the encryption:
$$ x = a^{-1} \cdot y - b\,({\rm mod}\; m)$$
where:
- $a^{-1}$ represents the multiplicative inverse of $a$

Remind that given a number $a$, the inverse $a^{-1}$ with respect to multiplication modulo $m$ is that number that
$$ a^{-1} \cdot a \equiv a \cdot a^{-1} \equiv 1\,({\rm mod}\;m)$$

In [ ]:
def affine_decrypt(ciphertext, a, b):
    ''' Decrypt `ciphertext` (str) as an Affine cipher with a given `key` made
    of two values `a` (int) and `b` (int) and return the corresponding
    plaintext (str)'''
    # CODE HERE
    return plaintext

In [ ]:
ciphertext = 'wniir praik!'
a, b = 3, 1

plaintext = affine_decrypt(ciphertext, a, b)
print(ciphertext, '->', plaintext)
# expected output 'wniir praik! -> hello world!'

### Breaking Cipher

In [ ]:
# Load ciphertext
with open(data_dir / 'ciphertext_affine.txt', mode='r', encoding='utf-8') as f:
    ciphertext = f.read()

print(ciphertext[:800])

In [ ]:
# CODE HERE

## Vigenère cipher

The cipher is named after **Blaise de Vigenère** (1523–1596), a French diplomat and cryptographer — but somewhat unfairly. The system was actually first described by **Giovan Battista Bellaso** in 1553. Vigenère invented a related but stronger autokey cipher. The misattribution stuck, largely due to later writers crediting him by mistake.

For roughly three centuries, the Vigenère cipher was considered unbreakable — earning the nickname "le chiffre indéchiffrable" (**the indecipherable cipher**). This reputation made it a go-to tool for sensitive diplomatic and military correspondence across Europe.

The myth of its invincibility was shattered in 1863 when Prussian military officer Friedrich Kasiski published a method to crack it — now called the **Kasiski test**. Notably, Charles Babbage had independently discovered the same method a few years earlier, around 1854, but never published it.

Despite being broken in theory, the Vigenère cipher continued to see real-world use. The Confederate Army used it during the American Civil War, employing brass cipher disks with keywords like "MANCHESTER BLUFF" and "COMPLETE VICTORY" — with poor results, as Union cryptanalysts routinely broke their messages. It was also used in various forms into the early 20th century, before being fully superseded by machine ciphers.

The Vigenère cipher marks a pivotal moment in cryptographic history — the shift from monoalphabetic to **polyalphabetic** thinking. Its eventual defeat drove the development of more sophisticated cryptanalysis, and the lesson it teaches — that periodic keys are a fatal weakness — directly influenced the design of the one-time pad and eventually modern stream ciphers.

### Encryption

Unlike the Affine or Caesar ciphers, where every letter is shifted by the same amount, the Vigenère cipher uses a keyword to apply a different shift to each letter of the plaintext. This makes frequency analysis much harder, since the same plaintext letter can map to many different ciphertext letters.

The encryption procedure is the following:
1. choose a keyword (e.g., "key")
2. repeat it to mathc the plaintext length
3. shift each letter of the plaintext by the index of the key letter in the alphabet

**Example**

| plaintext | key | shift | ciphertext |
|:---------:|:---:|------:|-----------:|
| h         |  K  |    10 | r          |
| e         |  E  |     4 | i          |
| l         |  Y  |    24 | j          |
| l         |  K  |    10 | v          |
| o         |  E  |     4 | s          |


Note that:
- Unlike Caesar or Affine in which there is a single, fixed mapping from the plaintext alphabet to the ciphertext alphabet (**monoalphabetic**), each plaintext letter can encrypt to multiple ciphertext letters (**polyalphabetic**), defeating simple frequency analysis. 
- Key length matters enormously. A short key is repetitive and vulnerable; a key as long as the message (used once) becomes a one-time pad which theoretically unbreakable.
- key space is effectively much larger than monoalphabetic ciphers, though not infinite.


In [ ]:
def vigenere_encrypt(plaintext, key):
    ''' Encrypt `plaintext` (str) as a Vigenere cipher with a given `key` (str)
    and return the corresponding ciphertext (str)'''
    # CODE HERE
    return ciphertext

In [ ]:
plaintext = 'hello world!'
key = 'key'

ciphertext = vigenere_encrypt(plaintext, key)
print(plaintext, '->', ciphertext)
# expected output hello world! -> rijvs uyvjn!

### Decryption

In [ ]:
def vigenere_decrypt(ciphertext, key):
    ''' Decrypt `ciphertext` (str) as a Vigenere cipher with a given `key` (str)
       and return the corresponding plaintext (str)'''
    # CODE HERE
    return ciphertext

In [ ]:
ciphertext = 'rijvs uyvjn!'
key = 'key'

plaintext = vigenere_decrypt(ciphertext, key)
print(ciphertext, '->', plaintext)
# expected output rijvs uyvjn! -> hello world!

### Ciphertext

In [ ]:
with open(data_dir / 'ciphertext_vigenere.txt', mode='r') as f:
    ciphertext = f.read()
print(ciphertext[:1000])

### Breaking Cipher (Homework)

In [ ]:
key_length = 6 # hint, key lenght is known!